# 02 — Technologies, pipeline et bonnes pratiques

Ce notebook explique **comment** la branche récupère les données, sans exiger de maîtriser Python.

---

## Wikidata ≠ Wikipedia

| | Wikipedia | Wikidata |
|--|-----------|----------|
| Contenu | Articles en prose | **Faits** structurés (ville X jumelée avec Y) |
| Accès | Pages HTML | **API SPARQL** (requêtes comme du SQL) |
| Pour nous | Difficile à scraper proprement | **Idéal** pour relations entre entités |

On utilise la propriété **`P190`** (= *twinned administrative body* / ville jumelée).

## Stack technique (branche data)

| Outil | Rôle |
|-------|------|
| **Python 3** | Langage des scripts |
| **Pandas** | Tableaux de données (CSV, upsert) |
| **SPARQLWrapper** | Client pour envoyer des requêtes SPARQL |
| **Git / GitHub** | Versionner code + données |
| **GitHub Actions** | Robot qui lance le script automatiquement |

Pas de base de données lourde : fichiers CSV/JSON suffisent pour un projet week-end.

## Être respectueux envers Wikidata (parcimonie)

Wikidata est un **service public** partagé. Notre script applique plusieurs règles :

1. **User-Agent identifié** — en-tête HTTP avec nom du projet + email :
   ```
   little-project-sister-cities/... (mailto:votre@email.com)
   ```
2. **Pagination** — lots de 5 000 lignes max par requête
3. **Pause 65 s** entre deux lots (évite le rate-limit 429)
4. **Retry** — en cas d'erreur, on attend au lieu de marteler le serveur
5. **Cron hebdomadaire** — pas de sync en continu

> Si tu lances le script toi-même, évite de le spammer : une fois suffit pour tester.

In [ ]:
# Lire le User-Agent enregistré lors du dernier run
import json
from pathlib import Path

meta_path = Path("..") / "data" / "processed" / "metadata.json"

if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    print("User-Agent du dernier run :")
    print(meta.get("user_agent", "(non renseigné)"))
    print()
    print(f"Dernier run : {meta.get('last_run')}")
    print(f"Mode       : {meta.get('sync_mode')}")
else:
    print("Pas encore de metadata.json — lance sync_wikidata.py d'abord.")

## Deux modes de synchronisation

### Mode **delta** (léger, par défaut)

- Lit `metadata.json` → champ **`last_run`** (date ISO du dernier succès)
- Requête SPARQL filtrée : entités modifiées **après** cette date (`schema:dateModified`)
- Fusionne seulement les **nouvelles lignes** dans le CSV (upsert)

### Mode **`--full`** (complet)

- Ignore `last_run`
- Télécharge **tous** les jumelages P190
- Utile au **premier run** ou en **filet de sécurité**

### Calendrier automatique (GitHub Actions)

| Quand | Mode |
|-------|------|
| Dimanche 03:00 UTC | delta |
| 1er du mois 04:00 UTC | **full** |
| Manuel (Actions ou terminal) | au choix |

**Pourquoi un full mensuel ?** Le delta peut rater un jumelage si Wikidata n'a pas mis à jour la date de l'entité. Le full mensuel rattrape ces cas.

## Stockage intelligent : pas de copie inutile

### 1. Clé unique `(ville_A_id, ville_B_id)`

Les IDs Wikidata (`Q90` = Paris) sont **triés** : Lyon–Paris et Paris–Lyon deviennent la **même ligne**.

### 2. Upsert Pandas

```python
# Simplifié :
fusion = pd.concat([ancien_csv, nouvelles_lignes])
fusion = fusion.drop_duplicates(subset=["ville_A_id", "ville_B_id"], keep="last")
```

→ On **met à jour** une paire existante au lieu de dupliquer.

### 3. Graphe JSON seulement si le CSV change

Le script compare un **hash SHA256** du CSV avant/après. Pas de changement → pas de regénération de `graphe_jumelages.json`.

### 4. Git : commit seulement si les fichiers data changent

Le workflow CI fait `git diff` avant de committer — pas de commit vide.

## Exemple de requête SPARQL (simplifiée)

Tu peux la tester sur https://query.wikidata.org/ :

```sparql
SELECT ?city ?cityLabel ?twin ?twinLabel
WHERE {
  ?city wdt:P190 ?twin .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
LIMIT 10
```

- `wdt:P190` = lien « est jumelée avec »
- `SERVICE wikibase:label` = récupère le nom lisible (en anglais ici)

## Lancer le pipeline à la main

Dans un terminal, à la racine du projet :

```bash
pip install -r requirements.txt

# Premier téléchargement (ou reset complet)
python scripts/sync_wikidata.py --full

# Mise à jour légère ensuite
python scripts/sync_wikidata.py

# Agrégats pour le site
python scripts/build_stats.py
```

**Variable optionnelle :**
```bash
set WIKIDATA_CONTACT_EMAIL=ton.email@etu.fr
```

## Limites à connaître (esprit critique)

- Wikidata est **incomplet** vs la réalité municipale
- Le delta filtre la **date de l'entité**, pas toujours la date d'ajout d'un jumelage
- Wikidata peut renvoyer **429** (trop de requêtes) — d'où les pauses
- Les données d'exemple du repo peuvent être **petites** en attendant un `--full` réussi

→ Toujours afficher la **source** et la **date** sur le site final.